In [1]:
#library(IRkernel)
#IRkernel::installspec(name = 'python_r_env', displayname = 'R (python_r_env)')
#Sys.which("R")

In [2]:
#================================================================================
# Load Libraries and Environment Variables
#================================================================================
library(dplyr)
library(here)
library(poLCA)
library(dotenv)
library(readxl)
library(writexl)

# Load environment variables from the .env file
env_file <- normalizePath("../../.env", mustWork = FALSE) # Ensure relative path works
if (!file.exists(env_file)) {
  stop(paste("Environment file not found at:", env_file))
}
load_dot_env(env_file)

# Define workspace path
WORKSPACE_PATH <- Sys.getenv("WORKSPACE_PATH")
WORKSPACE_PATH <- normalizePath(WORKSPACE_PATH)

# Source utility functions
source(file.path(WORKSPACE_PATH, "src/config/utils.R"))

DATA_DIR <- file.path(WORKSPACE_PATH, "data")
MOMENT_OF_SUICIDE_FEATURES <- split_string(Sys.getenv("MOMENT_OF_SUICIDE_FEATURES"))
SOCIO_DEMOGRAPHIC_FEATURES <- split_string(Sys.getenv("SOCIO_DEMOGRAPHIC_FEATURES"))



Dołączanie pakietu: 'dplyr'


Następujące obiekty zostały zakryte z 'package:stats':

    filter, lag


Następujące obiekty zostały zakryte z 'package:base':

    intersect, setdiff, setequal, union


here() starts at C:/Users/huber/OneDrive/Dokumenty/GitHub/ippan_suicide_study

Ładowanie wymaganego pakietu: scatterplot3d

Ładowanie wymaganego pakietu: MASS


Dołączanie pakietu: 'MASS'


Następujący obiekt został zakryty z 'package:dplyr':

    select



Dołączanie pakietu: 'openxlsx2'


Następujący obiekt został zakryty z 'package:writexl':

    write_xlsx


Następujący obiekt został zakryty z 'package:readxl':

    read_xlsx




In [3]:
excel_file_path <- file.path(DATA_DIR, "processed", "r_test.xlsx")

In [4]:
write_excel <- function(file_path, data, ...) {
  # Validate inputs
  if (!inherits(data, "data.frame")) {
    stop("The 'data' argument must be a data.frame or tibble.")
  }
  
  # Normalize and validate file path
  file_path <- normalizePath(file_path, winslash = "/", mustWork = FALSE)
  print(paste("Normalized file path:", file_path))
  
  if (!grepl("\\.xlsx$", file_path, ignore.case = TRUE)) {
    stop("The file path must end with '.xlsx'.")
  }

  # Check if the directory exists
  dir_path <- dirname(file_path)
  if (!dir.exists(dir_path)) {
    stop(paste("The directory does not exist:", dir_path))
  }

  # Extract additional arguments
  args <- list(...)
  sheet_name <- if (!is.null(args$sheet_name)) args$sheet_name else "Sheet1"
  if_sheet_exists <- if (!is.null(args$if_sheet_exists)) args$if_sheet_exists else "replace"
  print(paste("Sheet name:", sheet_name))
  print(paste("If sheet exists:", if_sheet_exists))
  
  # Check if file exists
  file_exists <- file.exists(file_path)
  print(paste("File exists:", file_exists))
  
  wb_data <- list()

  if (file_exists) {
    # Load existing workbook
    tryCatch({
      sheet_names <- readxl::excel_sheets(file_path)
      wb_data <- map(setNames(sheet_names, sheet_names), ~ readxl::read_excel(file_path, sheet = .x))
    }, error = function(e) {
      stop(paste("Failed to read existing Excel file. Error:", e$message))
    })
  }

  # Handle sheet existence
  if (sheet_name %in% names(wb_data)) {
    if (if_sheet_exists == "replace") {
      wb_data[[sheet_name]] <- data
    } else if (if_sheet_exists == "overlay") {
      wb_data[[sheet_name]] <- rbind(wb_data[[sheet_name]], data)
    } else {
      stop("Invalid 'if_sheet_exists' value. Use 'replace' or 'overlay'.")
    }
  } else {
    wb_data[[sheet_name]] <- data
  }
  
  # Write the data to the Excel file
  tryCatch({
    writexl::write_xlsx(wb_data, path = file_path)
    message(paste("Data successfully written to:", file_path, "in sheet:", sheet_name))
  }, error = function(e) {
    stop(paste("Failed to save workbook at:", file_path, "\nError:", e$message))
  })
}


In [5]:
# Nowy plik z arkuszem "Sheet1"
write_excel(excel_file_path, data.frame(a = 1:5, b = 6:10), sheet_name = "Sheet1")

# Dodanie nowego arkusza "Sheet2"
write_excel(excel_file_path, data.frame(c = 11:15, d = 16:20), sheet_name = "Sheet2")

# Nadpisanie danych w arkuszu "Sheet1"
write_excel(excel_file_path, data.frame(a = 21:25, b = 26:30), sheet_name = "Sheet1", if_sheet_exists = "replace")


[1] "Normalized file path: C:/Users/huber/OneDrive/Dokumenty/GitHub/ippan_suicide_study/data/processed/r_test.xlsx"
[1] "Sheet name: Sheet1"
[1] "If sheet exists: replace"
[1] "File exists: TRUE"


Data successfully written to: C:/Users/huber/OneDrive/Dokumenty/GitHub/ippan_suicide_study/data/processed/r_test.xlsx in sheet: Sheet1



[1] "Normalized file path: C:/Users/huber/OneDrive/Dokumenty/GitHub/ippan_suicide_study/data/processed/r_test.xlsx"
[1] "Sheet name: Sheet2"
[1] "If sheet exists: replace"
[1] "File exists: TRUE"


Data successfully written to: C:/Users/huber/OneDrive/Dokumenty/GitHub/ippan_suicide_study/data/processed/r_test.xlsx in sheet: Sheet2



[1] "Normalized file path: C:/Users/huber/OneDrive/Dokumenty/GitHub/ippan_suicide_study/data/processed/r_test.xlsx"
[1] "Sheet name: Sheet1"
[1] "If sheet exists: replace"
[1] "File exists: TRUE"


Data successfully written to: C:/Users/huber/OneDrive/Dokumenty/GitHub/ippan_suicide_study/data/processed/r_test.xlsx in sheet: Sheet1



In [6]:
.libPaths()

[1] "C:/Users/huber/anaconda3/envs/python_r_env/Lib/R/library"